# 05 Backtest and Regime Analysis

This notebook completes **Team Contribution 4: Evaluation lead** from the proposal:

- regime-segmented model evaluation
- simple long/flat backtest with transaction costs
- robustness checks
- model-failure case studies

It also reports the continuous next-day return modeling supplement generated by `scripts/evaluate_backtest.py`, because the proposal's modeling section included both binary direction and continuous return outcomes.

## 1. Setup and Reproducible Outputs

The evaluation script reads the outputs from notebook 04 and writes the tables used below. Running this cell will regenerate the evaluation files if they are missing.

In [ ]:
from pathlib import Path
import runpy

import pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'processed'
FIG = ROOT / 'figures'
REPORT = ROOT / 'report'

expected_outputs = [
    DATA / 'model_regime_metrics.csv',
    DATA / 'backtest_performance_summary.csv',
    DATA / 'backtest_regime_summary.csv',
    DATA / 'robustness_threshold_cost_summary.csv',
    DATA / 'model_failure_cases.csv',
    DATA / 'return_model_metrics_summary.csv',
]

if not all(path.exists() for path in expected_outputs):
    runpy.run_path(str(ROOT / 'scripts' / 'evaluate_backtest.py'), run_name='__main__')
else:
    print('Using existing generated evaluation outputs.')

In [ ]:
regime_metrics = pd.read_csv(DATA / 'model_regime_metrics.csv')
backtest_summary = pd.read_csv(DATA / 'backtest_performance_summary.csv')
backtest_regime = pd.read_csv(DATA / 'backtest_regime_summary.csv')
robustness = pd.read_csv(DATA / 'robustness_threshold_cost_summary.csv')
failure_cases = pd.read_csv(DATA / 'model_failure_cases.csv', parse_dates=['Date'])
return_metrics = pd.read_csv(DATA / 'return_model_metrics_summary.csv')

print('Loaded evaluation tables:')
for name, df in [
    ('regime_metrics', regime_metrics),
    ('backtest_summary', backtest_summary),
    ('backtest_regime', backtest_regime),
    ('robustness', robustness),
    ('failure_cases', failure_cases),
    ('return_metrics', return_metrics),
]:
    print(f'{name:20s} {df.shape}')

## 2. Regime-Segmented Evaluation

This directly addresses the proposal question: **Does predictive performance differ across calm and high-volatility market regimes?** The table below uses expanding-window out-of-sample predictions from the all-feature Logit and XGB models.

In [ ]:
oos_regime = regime_metrics[
    (regime_metrics['split'] == 'oos_expanding')
    & (regime_metrics['feature_set'] == 'all')
    & (regime_metrics['model_name'].isin(['Logit', 'XGB']))
].copy()

display(
    oos_regime[
        ['model_name', 'regime', 'n', 'up_day_rate', 'accuracy', 'auc', 'f1', 'avg_next_return', 'volatility']
    ].round(4)
)

Key reading: model performance is weak and unstable across regimes. The 2008-2009 crisis is especially difficult for both models, while XGB performs closest to random in COVID and only modestly above random in the broad `Other` regime.

## 3. Long/Flat Backtest with Transaction Costs

The strategy is simple and intentionally conservative:

- long S&P 500 next day when predicted up-day probability is at least 0.50
- flat otherwise
- subtract 1 basis point whenever the signal changes
- compare against buy-and-hold over the same dates

In [ ]:
oos_backtest = backtest_summary[
    (backtest_summary['split'] == 'oos_expanding')
    & (backtest_summary['feature_set'] == 'all')
    & (backtest_summary['model_name'].isin(['Logit', 'XGB']))
].copy()

display(
    oos_backtest[
        [
            'model_name',
            'strategy_cumulative_return',
            'buy_hold_cumulative_return',
            'strategy_annualized_return',
            'strategy_sharpe',
            'strategy_max_drawdown',
            'exposure',
            'trades',
        ]
    ].round(4)
)

In [ ]:
display(Image(filename=str(FIG / 'fig6_backtest_equity_curves.png')))

The strategy produces positive cumulative returns, but it does **not** beat buy-and-hold over the full expanding-window sample. This weakens the economic-significance claim and should be reported honestly.

## 4. Backtest by Market Regime

The regime-level backtest checks whether strategy value is concentrated in only one market environment.

In [ ]:
oos_backtest_regime = backtest_regime[
    (backtest_regime['split'] == 'oos_expanding')
    & (backtest_regime['feature_set'] == 'all')
    & (backtest_regime['model_name'].isin(['Logit', 'XGB']))
].copy()

display(
    oos_backtest_regime[
        [
            'model_name',
            'regime',
            'strategy_cumulative_return',
            'buy_hold_cumulative_return',
            'strategy_sharpe',
            'strategy_max_drawdown',
            'exposure',
            'trades',
        ]
    ].round(4)
)

In [ ]:
display(Image(filename=str(FIG / 'fig7_regime_backtest_returns.png')))

## 5. Robustness Checks

The table below tests whether conclusions depend strongly on the trading threshold or transaction-cost assumption. This guards against over-interpreting one arbitrary 0.50 threshold.

In [ ]:
display(
    robustness.sort_values('cumulative_return', ascending=False)[
        ['model_name', 'threshold', 'transaction_cost', 'cumulative_return', 'sharpe', 'max_drawdown', 'exposure', 'trades']
    ].round(4)
)

In [ ]:
display(Image(filename=str(FIG / 'fig8_backtest_robustness.png')))

The robustness check shows sensitivity to both threshold and trading costs. This means the backtest should be treated as exploratory economic evidence, not as proof of a stable trading rule.

## 6. Model-Failure Case Studies

These cases are selected for report discussion. They show dates where the model either went long before a loss, missed a rally, or made a high-confidence classification error.

In [ ]:
display(
    failure_cases[
        [
            'failure_type',
            'Date',
            'regime',
            'model_name',
            'y_pred_proba',
            'return_next_day',
            'headline_count',
            'vix',
            'daily_text_snippet',
        ]
    ].head(15).round(4)
)

For the final write-up, pick a few rows from `model_failure_cases.csv` and explain why headline sentiment may not capture sudden reversals, crisis rebounds, or macro-driven market moves.

## 7. Continuous Return Modeling Supplement

This section fills the proposal's continuous next-day return modeling requirement. It should be discussed as a modeling supplement rather than as the core backtest engine.

In [ ]:
test_return_metrics = return_metrics[return_metrics['split'] == 'test'].copy()
display(
    test_return_metrics.sort_values('information_coefficient', ascending=False)[
        ['model', 'feature_set', 'rmse', 'mae', 'r2', 'information_coefficient', 'direction_accuracy']
    ].round(5)
)

In [ ]:
display(Image(filename=str(FIG / 'fig9_return_model_predictions.png')))

The return models have weak out-of-sample explanatory power. The best test-set information coefficient is small and R-squared is negative, so the continuous-return results should be framed as robustness evidence rather than as a strong prediction result.

## 8. Evaluation-Lead Takeaways

- Predictive performance varies across regimes and is weakest in crisis-like periods.
- The long/flat strategy earns positive returns in the expanding-window test but does not outperform buy-and-hold.
- Backtest results are sensitive to thresholds and transaction costs.
- Failure cases show that headline sentiment can miss sharp rebounds and macro-driven moves.
- Continuous return models add the planned regression-style outcome, but their test-set fit is weak.

Overall, the evaluation supports a cautious conclusion: financial-news sentiment contains limited, unstable signal, but the evidence is not strong enough to claim reliable trading value.